# **Build Unified Document Dataset**
This notebook combines the metadata generated by the four agency crawlers into a single standardized dataset.

## Import Required Libraries

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
!pip install pdfplumber
import pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 52.1 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load and Verify Metadata

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/newstart_ai")

METADATA_DIR = BASE_DIR / "data" / "metadata"

uscis_df = pd.read_csv(METADATA_DIR / "uscis_metadata.csv")
dmv_df   = pd.read_csv(METADATA_DIR / "dmv_metadata.csv")
ssa_df   = pd.read_csv(METADATA_DIR / "ssa_metadata.csv")
irs_df   = pd.read_csv(METADATA_DIR / "irs_metadata.csv")

print("USCIS:", uscis_df.shape)
print("DMV   :", dmv_df.shape)
print("SSA   :", ssa_df.shape)
print("IRS   :", irs_df.shape)

USCIS: (256, 5)
DMV   : (277, 4)
SSA   : (198, 5)
IRS   : (23, 5)


In [ ]:
datasets = {
    "USCIS": uscis_df,
    "DMV": dmv_df,
    "SSA": ssa_df,
    "IRS": irs_df
}

for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print(df.columns.tolist())

USCIS
['filename', 'filepath', 'agency', 'form_number', 'document_type']
DMV
['filename', 'filepath', 'agency', 'document_type']
SSA
['filename', 'filepath', 'agency', 'form_number', 'document_type']
IRS
['filename', 'agency', 'form_number', 'document_type', 'url']


## Standardize Metadata

Each agency procided slightly different information for the metadata, which led to me creating different fields for each. This section converts every dataset into a consistent schema so that all agencies can be combined into a single dataframe.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/newstart_ai")

def normalize_metadata(df, agency):
    """
    Standardize metadata schema across all agencies.
    """

    df = df.copy()

    # Required columns
    required_columns = [
        "filename",
        "filepath",
        "agency",
        "form_number",
        "document_type",
        "url"
    ]

    # Add missing columns
    for col in required_columns:
        if col not in df.columns:
            df[col] = ""

    # Build filepath if missing (IRS)
    if agency == "IRS":
        df["filepath"] = (
            BASE_DIR /
            "data" /
            "raw" /
            "irs"
        ).as_posix() + "/" + df["filename"]

    # DMV has no form numbers
    if agency == "DMV":
        df["form_number"] = ""

    # Ensure agency name is consistent
    df["agency"] = agency

    # Reorder columns
    df = df[
        [
            "filename",
            "filepath",
            "agency",
            "form_number",
            "document_type",
            "url",
        ]
    ]

    return df

In [ ]:
uscis_df = normalize_metadata(uscis_df, "USCIS")
dmv_df   = normalize_metadata(dmv_df, "DMV")
ssa_df   = normalize_metadata(ssa_df, "SSA")
irs_df   = normalize_metadata(irs_df, "IRS")

In [ ]:
for name, df in {
    "USCIS": uscis_df,
    "DMV": dmv_df,
    "SSA": ssa_df,
    "IRS": irs_df,
}.items():

    print("=" * 70)
    print(name)
    print(df.head(2))

USCIS
         filename                                           filepath agency  \
0     i-765ws.pdf  /content/drive/MyDrive/newstart_ai/data/raw/us...  USCIS   
1  i-361instr.pdf  /content/drive/MyDrive/newstart_ai/data/raw/us...  USCIS   

  form_number document_type url  
0       I-765          form      
1       I-361  instructions      
DMV
                                            filename  \
0  statement-of-multiple-county-use-of-vehicle-re...   
1  traffic-violator-school-foreign-language-appro...   

                                            filepath agency form_number  \
0  /content/drive/MyDrive/newstart_ai/data/raw/dm...    DMV               
1  /content/drive/MyDrive/newstart_ai/data/raw/dm...    DMV               

  document_type url  
0          form      
1          form      
SSA
             filename                                           filepath  \
0   CMS-1763-508C.pdf  /content/drive/MyDrive/newstart_ai/data/raw/ss...   
1  CMS-L564_SP508.pdf  /content/d

## Handle Missing Values

In [ ]:
# Replace NaN with empty strings in all datasets

uscis_df = uscis_df.fillna("")
dmv_df   = dmv_df.fillna("")
ssa_df   = ssa_df.fillna("")
irs_df   = irs_df.fillna("")

In [ ]:
print(ssa_df.head())

                       filename  \
0             CMS-1763-508C.pdf   
1            CMS-L564_SP508.pdf   
2                  SSA-8-SP.pdf   
3  cms-40b-508c-2025-rev-38.pdf   
4            cms-40b-s-508c.pdf   

                                            filepath agency form_number  \
0  /content/drive/MyDrive/newstart_ai/data/raw/ss...    SSA               
1  /content/drive/MyDrive/newstart_ai/data/raw/ss...    SSA               
2  /content/drive/MyDrive/newstart_ai/data/raw/ss...    SSA    SSA-8-SP   
3  /content/drive/MyDrive/newstart_ai/data/raw/ss...    SSA               
4  /content/drive/MyDrive/newstart_ai/data/raw/ss...    SSA               

  document_type url  
0          form      
1          form      
2          form      
3          form      
4          form      


## Combine All Agency Metadata
Merge the standardized metadata from USCIS, DMV, SSA, and IRS into one unified dataset that will be used throughout the remainder of the project.

In [ ]:
metadata_df = pd.concat(
    [
        uscis_df,
        dmv_df,
        ssa_df,
        irs_df
    ],
    ignore_index=True
)

print("Rows:", len(metadata_df))
print("Columns:", metadata_df.columns.tolist())

metadata_df.head()

Rows: 754
Columns: ['filename', 'filepath', 'agency', 'form_number', 'document_type', 'url']


,filename,filepath,agency,form_number,document_type,url
0,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,
1,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,
2,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,
3,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,
4,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,


## Verify Combined Dataset

Review the merged dataset to confirm that documents from each agency were successfully included.

In [ ]:
metadata_df["agency"].value_counts()

,count
agency,
DMV,277
USCIS,256
SSA,198
IRS,23


In [ ]:
duplicates = metadata_df["filename"].duplicated().sum()

print("Duplicate filenames:", duplicates)

Duplicate filenames: 0


In [ ]:
from pathlib import Path

metadata_df["file_exists"] = metadata_df["filepath"].apply(
    lambda x: Path(x).exists()
)

metadata_df["file_exists"].value_counts()

,count
file_exists,
True,754


## Extract Text from PDF Documents

Since machine learning models require textual input rather than PDF files, this section defines helper functions that extract readable text from each document

In [ ]:
import pdfplumber

def extract_pdf_text(pdf_path):
    """
    Extract text from a PDF using pdfplumber.

    Returns:
        text (str): Extracted text
    """

    text = ""

    try:
        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")

    return text.strip()

In [ ]:
sample_path = metadata_df.iloc[0]["filepath"]

print(sample_path)

sample_text = extract_pdf_text(sample_path)

print(sample_text[:2000])

/content/drive/MyDrive/newstart_ai/data/raw/uscis/i-765ws.pdf
Form I-765 Worksheet USCIS
Form I-765WS
Department of Homeland Security
OMB No. 1615-0040
U.S. Citizenship and Immigration Services Expires 08/31/2027
If you are applying for employment authorization under the (c)(14), Deferred Action, or (c)(33), Consideration of Deferred Action for
Childhood Arrivals, categories, you must complete this worksheet so we can determine whether you have an economic need to work.
In the spaces provided, indicate your current annual income, your current annual expenses, and the total current value of your assets.
Supporting evidence is not required, but U.S. Citizenship and Immigration Services (USCIS) will accept and review any documentation
that you submit. You do not need to include other household members' financial information to establish your own economic necessity.
Part 1. Your Full Name
1.a. Family Name
(Last Name)
1.b. Given Name
(First Name)
1.c. Middle Name
Part 2. Financial Informati

## Reusable Extraction Function
This pulls the document text, page count, and extraction status. This will be helpful for the final ml dataset

In [ ]:
from tqdm.auto import tqdm
import pdfplumber


def extract_pdf_data(pdf_path):
    """
    Extract text and metadata from a PDF.

    Returns:
        text
        page_count
        success
    """

    text = ""
    page_count = 0

    try:
        with pdfplumber.open(pdf_path) as pdf:

            page_count = len(pdf.pages)

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

        return text.strip(), page_count, True

    except Exception as e:
        print(f"Failed: {pdf_path}")
        print(e)

        return "", 0, False

In [ ]:
test_results = []

for path in metadata_df["filepath"].head(5):

    text, pages, success = extract_pdf_data(path)

    test_results.append({
        "filepath": path,
        "text_length": len(text),
        "pages": pages,
        "success": success
    })


pd.DataFrame(test_results)

,filepath,text_length,pages,success
0,/content/drive/MyDrive/newstart_ai/data/raw/us...,1281,1,True
1,/content/drive/MyDrive/newstart_ai/data/raw/us...,17876,6,True
2,/content/drive/MyDrive/newstart_ai/data/raw/us...,9966,4,True
3,/content/drive/MyDrive/newstart_ai/data/raw/us...,11552,4,True
4,/content/drive/MyDrive/newstart_ai/data/raw/us...,14082,8,True


## Build the Final Dataset

Process every document in the collection and augment the metadata with extracted text and document statistics.

In [ ]:
from tqdm.auto import tqdm

# Create a copy so we preserve metadata_df
final_df = metadata_df.copy()

# Lists to store results
texts = []
page_counts = []
success_flags = []

for path in tqdm(final_df["filepath"], desc="Extracting PDFs"):

    text, pages, success = extract_pdf_data(path)

    texts.append(text)
    page_counts.append(pages)
    success_flags.append(success)


# Add extracted information
final_df["text"] = texts
final_df["page_count"] = page_counts
final_df["text_length"] = final_df["text"].apply(len)
final_df["extraction_success"] = success_flags


final_df.head()

Extracting PDFs:   0%|          | 0/754 [00:00<?, ?it/s]

,filename,filepath,agency,form_number,document_type,url,file_exists,text,page_count,text_length,extraction_success
0,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,,True,Form I-765 Worksheet USCIS\nForm I-765WS\nDepa...,1,1281,True
1,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,,True,Instructions for Affidavit of\nFinancial Suppo...,6,17876,True
2,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,,True,"Instructions for Waiver of Certain Rights,\nPr...",4,9966,True
3,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,,True,Instructions for Request for Certificate of No...,4,11552,True
4,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,,True,Nonimmigrant Petition Based on Blanket L Petit...,8,14082,True


## Review Extraction Results

This verifies that the extraction pipeline completed successfully and checks if any documents nee more inspection

In [ ]:
final_df["extraction_success"].value_counts()

,count
extraction_success,
True,754


In [ ]:
final_df[[
    "agency",
    "filename",
    "text_length",
    "page_count"
]].head(10)

,agency,filename,text_length,page_count
0,USCIS,i-765ws.pdf,1281,1
1,USCIS,i-361instr.pdf,17876,6
2,USCIS,i-508instr.pdf,9966,4
3,USCIS,g-1566instr.pdf,11552,4
4,USCIS,i-129s.pdf,14082,8
5,USCIS,n-470.pdf,11858,7
6,USCIS,g-28iinstr.pdf,14295,4
7,USCIS,i-600ainstr.pdf,36600,11
8,USCIS,i-800asup2.pdf,3324,2
9,USCIS,i-129cwrinstr.pdf,19068,6


In [ ]:
final_df["text_length"].describe()

,text_length
count,754.000000
mean,15094.352785
std,28958.520523
min,112.000000
25%,3773.250000
50%,7638.000000
75%,19048.000000
max,630280.000000


In [ ]:
empty_docs = final_df[
    final_df["text_length"] == 0
]

print("Empty documents:", len(empty_docs))

empty_docs[[
    "filename",
    "agency"
]]

Empty documents: 0


,filename,agency


## Export Processed Dataset

In [ ]:
OUTPUT_PATH = (
    "/content/drive/MyDrive/newstart_ai/data/processed/final_dataset.csv"
)

final_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:")
print(OUTPUT_PATH)

Saved:
/content/drive/MyDrive/newstart_ai/data/processed/final_dataset.csv


In [ ]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 754 entries, 0 to 753
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   filename            754 non-null    object
 1   filepath            754 non-null    object
 2   agency              754 non-null    object
 3   form_number         754 non-null    object
 4   document_type       754 non-null    object
 5   url                 754 non-null    object
 6   file_exists         754 non-null    bool  
 7   text                754 non-null    object
 8   page_count          754 non-null    int64 
 9   text_length         754 non-null    int64 
 10  extraction_success  754 non-null    bool  
dtypes: bool(2), int64(2), object(7)
memory usage: 54.6+ KB


In [ ]:
final_df.head(3)

,filename,filepath,agency,form_number,document_type,url,file_exists,text,page_count,text_length,extraction_success
0,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,,True,Form I-765 Worksheet USCIS\nForm I-765WS\nDepa...,1,1281,True
1,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,,True,Instructions for Affidavit of\nFinancial Suppo...,6,17876,True
2,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,,True,"Instructions for Waiver of Certain Rights,\nPr...",4,9966,True


In [ ]:
final_df["agency"].value_counts()

,count
agency,
DMV,277
USCIS,256
SSA,198
IRS,23


In [ ]:
final_df.groupby("agency")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
agency,,,,,,,,
DMV,277.0,4479.599278,4009.331751,112.0,1900.00,3355.0,5694.00,34643.0
IRS,23.0,63052.173913,131830.186837,3326.0,8913.50,23338.0,49248.50,630280.0
SSA,198.0,10875.404040,7137.052613,1749.0,5306.25,7724.5,16433.75,41135.0
USCIS,256.0,25534.230469,21612.786536,665.0,11162.50,21327.5,31791.00,148353.0


In [ ]:
short_docs = final_df[
    final_df["text_length"] < 100
]

print("Documents under 100 characters:", len(short_docs))

short_docs[
    [
        "filename",
        "agency",
        "text_length"
    ]
]

Documents under 100 characters: 0


,filename,agency,text_length


In [ ]:
metadata_output = "/content/drive/MyDrive/newstart_ai/data/processed/final_metadata.csv"

metadata_df.to_csv(
    metadata_output,
    index=False
)

print(metadata_output)

/content/drive/MyDrive/newstart_ai/data/processed/final_metadata.csv


## Add Document Identifiers
Assigned each document a unique ID so that metadata and extracted text can stay synched

In [ ]:
import pandas as pd

dataset_path = "/content/drive/MyDrive/newstart_ai/data/processed/final_dataset.csv"

final_df = pd.read_csv(dataset_path)

final_df.head()

,filename,filepath,agency,form_number,document_type,url,file_exists,text,page_count,text_length,extraction_success
0,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,NaN,True,Form I-765 Worksheet USCIS\nForm I-765WS\nDepa...,1,1281,True
1,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,NaN,True,Instructions for Affidavit of\nFinancial Suppo...,6,17876,True
2,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,NaN,True,"Instructions for Waiver of Certain Rights,\nPr...",4,9966,True
3,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,NaN,True,Instructions for Request for Certificate of No...,4,11552,True
4,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,NaN,True,Nonimmigrant Petition Based on Blanket L Petit...,8,14082,True


In [ ]:
"document_id" in final_df.columns

False

In [ ]:
final_df.insert(
    0,
    "document_id",
    range(1, len(final_df) + 1)
)

final_df.head()

,document_id,filename,filepath,agency,form_number,document_type,url,file_exists,text,page_count,text_length,extraction_success
0,1,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,NaN,True,Form I-765 Worksheet USCIS\nForm I-765WS\nDepa...,1,1281,True
1,2,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,NaN,True,Instructions for Affidavit of\nFinancial Suppo...,6,17876,True
2,3,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,NaN,True,"Instructions for Waiver of Certain Rights,\nPr...",4,9966,True
3,4,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,NaN,True,Instructions for Request for Certificate of No...,4,11552,True
4,5,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,NaN,True,Nonimmigrant Petition Based on Blanket L Petit...,8,14082,True


In [ ]:
final_df.to_csv(
    dataset_path,
    index=False
)

print("Updated dataset saved!")
print(dataset_path)

Updated dataset saved!
/content/drive/MyDrive/newstart_ai/data/processed/final_dataset.csv


In [ ]:
check_df = pd.read_csv(dataset_path)

check_df.head()

,document_id,filename,filepath,agency,form_number,document_type,url,file_exists,text,page_count,text_length,extraction_success
0,1,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,NaN,True,Form I-765 Worksheet USCIS\nForm I-765WS\nDepa...,1,1281,True
1,2,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,NaN,True,Instructions for Affidavit of\nFinancial Suppo...,6,17876,True
2,3,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,NaN,True,"Instructions for Waiver of Certain Rights,\nPr...",4,9966,True
3,4,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,NaN,True,Instructions for Request for Certificate of No...,4,11552,True
4,5,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,NaN,True,Nonimmigrant Petition Based on Blanket L Petit...,8,14082,True


In [ ]:
metadata_path = "/content/drive/MyDrive/newstart_ai/data/processed/final_metadata.csv"

metadata_df = pd.read_csv(metadata_path)

metadata_df.head()

,filename,filepath,agency,form_number,document_type,url,file_exists
0,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,NaN,True
1,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,NaN,True
2,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,NaN,True
3,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,NaN,True
4,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,NaN,True


In [ ]:
metadata_df.insert(
    0,
    "document_id",
    range(1, len(metadata_df)+1)
)

metadata_df.head()

,document_id,filename,filepath,agency,form_number,document_type,url,file_exists
0,1,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,NaN,True
1,2,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,NaN,True
2,3,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,NaN,True
3,4,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,NaN,True
4,5,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,NaN,True


In [ ]:
metadata_df.to_csv(
    metadata_path,
    index=False
)

print("Metadata updated!")

Metadata updated!
